# **Chapter 2: The calendar layer**

# **Loca-T demand forecasting: Chapter 2, the calendar layer**

## Storyline
International arrivals (Chapter 1) show the yearly shape of demand from abroad, but
most of Loca-T's guests are Kenyan. Domestic travel follows the calendar: families
travel when schools are out, workers when a public holiday makes a long weekend.

We build a **daily calendar table** from 1 Jan 2024 to 31 Mar 2027. This first step
adds school breaks. A day is "out of school" from the day after a term closes to
the day before the next opens, weekends included.

## Source and honesty
Dates come from the Ministry of Education's published calendars (2024, 2025, 2026
and 2027) as reported in the press. Outlets disagree on some details, so we use the
version most repeat. The February 2027 half-term is marked **unverified** because
reports conflict. Term 1 2027 opening on 4 January is consistent across sources.

## Checks
- No two breaks overlap.
- December holidays should be about 10 weeks.
- Roughly a third of all days should be out of school.

In [1]:
from pathlib import Path

import pandas as pd

# (first day out of school, last day out of school, label)
SCHOOL_OUT = [
    # 2024
    ("2024-01-01", "2024-01-07", "december_holiday"),
    ("2024-02-29", "2024-03-03", "half_term"),
    ("2024-04-06", "2024-04-28", "april_holiday"),
    ("2024-06-20", "2024-06-23", "half_term"),
    ("2024-08-03", "2024-08-25", "august_holiday"),
    ("2024-10-26", "2025-01-05", "december_holiday"),
    # 2025
    ("2025-02-26", "2025-03-02", "half_term"),
    ("2025-04-05", "2025-04-27", "april_holiday"),
    ("2025-06-25", "2025-06-29", "half_term"),
    ("2025-08-02", "2025-08-24", "august_holiday"),
    ("2025-10-25", "2026-01-04", "december_holiday"),
    # 2026
    ("2026-02-25", "2026-03-01", "half_term"),
    ("2026-04-03", "2026-04-26", "april_holiday"),
    ("2026-06-24", "2026-06-28", "half_term"),
    ("2026-08-01", "2026-08-23", "august_holiday"),
    ("2026-10-24", "2027-01-03", "december_holiday"),
    # 2027 (Term 1 opens 4 Jan; the February half-term is UNVERIFIED)
    ("2027-02-19", "2027-02-28", "half_term_unverified"),
]

breaks = pd.DataFrame(SCHOOL_OUT, columns=["start", "end", "label"])
breaks["start"] = pd.to_datetime(breaks["start"])
breaks["end"] = pd.to_datetime(breaks["end"])
breaks["days"] = (breaks["end"] - breaks["start"]).dt.days + 1
breaks["weeks"] = (breaks["days"] / 7).round(1)
print(breaks.to_string(index=False))

# Check 1: no overlaps
ordered = breaks.sort_values("start").reset_index(drop=True)
overlaps = (ordered["start"].iloc[1:].values <= ordered["end"].iloc[:-1].values).sum()
print(f"\noverlapping breaks: {overlaps}")

# Daily table
days = pd.date_range("2024-01-01", "2027-03-31", freq="D")
calendar = pd.DataFrame({"date": days})
calendar["school_break"] = ""
for start, end, label in SCHOOL_OUT:
    mask = (calendar["date"] >= start) & (calendar["date"] <= end)
    calendar.loc[mask, "school_break"] = label
calendar["school_out"] = (calendar["school_break"] != "").astype(int)

# Check 3: share of days out of school, by year
by_year = calendar.groupby(calendar["date"].dt.year)["school_out"].agg(["sum", "count"])
by_year["share_out"] = (by_year["sum"] / by_year["count"]).round(3)
print("\nDays out of school by year:")
print(by_year.to_string())

out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
calendar.to_csv(out_dir / "calendar_school.csv", index=False)
print("\nsaved:", out_dir / "calendar_school.csv", "| rows:", len(calendar))

     start        end                label  days  weeks
2024-01-01 2024-01-07     december_holiday     7    1.0
2024-02-29 2024-03-03            half_term     4    0.6
2024-04-06 2024-04-28        april_holiday    23    3.3
2024-06-20 2024-06-23            half_term     4    0.6
2024-08-03 2024-08-25       august_holiday    23    3.3
2024-10-26 2025-01-05     december_holiday    72   10.3
2025-02-26 2025-03-02            half_term     5    0.7
2025-04-05 2025-04-27        april_holiday    23    3.3
2025-06-25 2025-06-29            half_term     5    0.7
2025-08-02 2025-08-24       august_holiday    23    3.3
2025-10-25 2026-01-04     december_holiday    72   10.3
2026-02-25 2026-03-01            half_term     5    0.7
2026-04-03 2026-04-26        april_holiday    24    3.4
2026-06-24 2026-06-28            half_term     5    0.7
2026-08-01 2026-08-23       august_holiday    23    3.3
2026-10-24 2027-01-03     december_holiday    72   10.3
2027-02-19 2027-02-28 half_term_unverified    10

## **Public holidays: look first, then trust**

Public holidays create long weekends, which matter most for domestic short trips.
Rather than typing dates by hand, we ask the `holidays` Python package for Kenya's
holidays between 1 Jan 2024 and 31 Mar 2027, and **cross-check** it against the
core 2026 dates that every web source we found agreed on.

Things to look for in the printout:
1. **Missing or extra dates** versus the web sources.
2. **Observed days:** in Kenya, a holiday that falls on a Sunday moves to the Monday,
   and the package may list that as a separate entry.
3. **Islamic holidays** (Eid) depend on the moon sighting, so any date the package
   gives for 2027 is an estimate, not an official date.
4. **Naming:** holiday names changed in recent years, but names do not matter for
   the model, only dates do.

No flags are built in this cell. We only look.

In [2]:
import holidays

print("holidays package version:", holidays.__version__)

rows = []
for year in range(2024, 2028):
    for d, name in holidays.country_holidays("KE", years=year).items():
        d = pd.Timestamp(d)
        if pd.Timestamp("2024-01-01") <= d <= pd.Timestamp("2027-03-31"):
            rows.append({"date": d, "weekday": d.day_name()[:3], "name": name})

pkg = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
print(pkg.to_string(index=False))

core_2026 = pd.to_datetime([
    "2026-01-01", "2026-04-03", "2026-04-06", "2026-05-01", "2026-06-01",
    "2026-10-20", "2026-12-12", "2026-12-25", "2026-12-26",
])
pkg_dates = set(pkg["date"])
print("\nCore 2026 dates from the web sources that the package does NOT have:",
      [d.date() for d in core_2026 if d not in pkg_dates])

extras = pkg[(pkg["date"].dt.year == 2026) & (~pkg["date"].isin(core_2026))]
print("\n2026 dates only the package has:")
print(extras.to_string(index=False) if len(extras) else "none")

holidays package version: 0.93
      date weekday                              name
2024-01-01     Mon                    New Year's Day
2024-03-29     Fri                       Good Friday
2024-04-01     Mon                     Easter Monday
2024-04-10     Wed                       Eid-al-Fitr
2024-05-01     Wed                         Labor Day
2024-05-10     Fri         National Tree Growing Day
2024-06-01     Sat                      Madaraka Day
2024-10-10     Thu                     Utamaduni Day
2024-10-20     Sun                      Mashujaa Day
2024-10-21     Mon           Mashujaa Day (observed)
2024-11-01     Fri                  Inauguration Day
2024-12-12     Thu                       Jamhuri Day
2024-12-25     Wed                     Christmas Day
2024-12-26     Thu                        Boxing Day
2025-01-01     Wed                    New Year's Day
2025-03-30     Sun           Eid-al-Fitr (estimated)
2025-03-31     Mon Eid-al-Fitr (observed, estimated)
2025-04-18     

## **Public holidays, long weekends and Easter as daily flags**

**Source:** the `holidays` package (Kenya), cross-checked in the previous cell. Every
core 2026 date matched the web sources. Eid dates are estimates (moon sighting), so
we carry an `holiday_estimated` flag instead of hiding the uncertainty.

**Definitions**
- `is_public_holiday`: any date in the package's Kenyan list, including observed days.
- `non_working`: weekend or public holiday.
- **Long weekend:** a run of 3 or more consecutive non-working days that contains at
  least one public holiday. `long_weekend_len` holds the length of the run.
- `easter_period`: from the Thursday before Good Friday to Easter Monday.

**Built on an extended range** (1 Dec 2023 to 30 Apr 2027) and then trimmed, so a long
weekend that straddles the edge of our window is not cut short.

**Limits:** we do not model "bridge" days (for example, a Tuesday holiday that makes
many people take the Monday off), and holidays that fall on a Saturday do not move.

**Checks:** the Christmas 2026 weekend should be 3 days (Fri 25 to Sun 27 Dec),
Easter 2026 should be 4 days (Fri 3 to Mon 6 Apr), and long weekends per year should
be in the single digits.

In [5]:
import numpy as np

# 1) Holidays for an extended range (so edge long weekends are not truncated)
hol_map, estimated_dates, good_fridays = {}, set(), []
for year in range(2023, 2028):
    for d, name in holidays.country_holidays("KE", years=year).items():
        d = pd.Timestamp(d)
        hol_map.setdefault(d, []).append(name)
        if "estimated" in name:
            estimated_dates.add(d)
        if name == "Good Friday":
            good_fridays.append(d)

ext = pd.DataFrame({"date": pd.date_range("2023-12-01", "2027-04-30", freq="D")})
ext["holiday_name"] = ext["date"].map(lambda d: " / ".join(hol_map.get(d, [])))
ext["is_public_holiday"] = (ext["holiday_name"] != "").astype(int)
ext["holiday_estimated"] = ext["date"].isin(estimated_dates).astype(int)
ext["dow"] = ext["date"].dt.dayofweek          # Monday = 0
ext["is_weekend"] = (ext["dow"] >= 5).astype(int)
ext["non_working"] = ((ext["is_weekend"] == 1) | (ext["is_public_holiday"] == 1)).astype(int)

# 2) Runs of consecutive non-working days that contain a public holiday
run_id = (ext["non_working"] != ext["non_working"].shift()).cumsum()
run_len = ext.groupby(run_id)["non_working"].transform("sum")
run_has_holiday = ext.groupby(run_id)["is_public_holiday"].transform("max")
is_lw = (ext["non_working"] == 1) & (run_len >= 3) & (run_has_holiday == 1)
ext["long_weekend_len"] = np.where(is_lw, run_len, 0)
ext["is_long_weekend"] = is_lw.astype(int)

# 3) Easter period: Thursday before Good Friday to Easter Monday
ext["easter_period"] = 0
for gf in good_fridays:
    mask = (ext["date"] >= gf - pd.Timedelta(days=1)) & (ext["date"] <= gf + pd.Timedelta(days=3))
    ext.loc[mask, "easter_period"] = 1

# 4) Trim to our window and add to the school-break calendar (rebuilt from the saved file)
window = (ext["date"] >= "2024-01-01") & (ext["date"] <= "2027-03-31")
flag_cols = ["date", "holiday_name", "is_public_holiday", "holiday_estimated", "dow",
             "is_weekend", "non_working", "is_long_weekend", "long_weekend_len", "easter_period"]
school = pd.read_csv("../data/processed/calendar_school.csv", parse_dates=["date"])
calendar_full = school.merge(ext.loc[window, flag_cols], on="date", how="left")

# 5) Table of long weekends (one row per run)
lw_rows = ext[is_lw].copy()
lw_rows["run"] = run_id[is_lw]
long_weekends = (
    lw_rows.groupby("run")
    .agg(start=("date", "min"), end=("date", "max"), days=("date", "count"),
         holidays=("holiday_name", lambda s: ", ".join(sorted({x for x in s if x}))))
    .reset_index(drop=True)
)
long_weekends = long_weekends[(long_weekends["end"] >= "2024-01-01") & (long_weekends["start"] <= "2027-03-31")]
long_weekends["start_day"] = long_weekends["start"].dt.day_name().str[:3]

print("Public holidays by year (weekday vs weekend):")
ph = calendar_full[calendar_full["is_public_holiday"] == 1]
print(ph.groupby([ph["date"].dt.year, ph["is_weekend"]]).size().unstack(fill_value=0)
        .rename(columns={0: "on weekday", 1: "on weekend"}).to_string())

print("\nLong weekends per year:")
print(long_weekends.groupby(long_weekends["start"].dt.year).size().to_string())

print("\nLong weekends from November 2026:")
print(long_weekends[long_weekends["start"] >= "2026-11-01"]
      [["start", "start_day", "end", "days", "holidays"]].to_string(index=False))

print("\nAll long weekends 2026:")
print(long_weekends[long_weekends["start"].dt.year == 2026]
      [["start", "start_day", "end", "days", "holidays"]].to_string(index=False))

print("\nEaster period days flagged:", int(calendar_full["easter_period"].sum()))

Public holidays by year (weekday vs weekend):
is_weekend  on weekday  on weekend
date                              
2024                12           2
2025                11           2
2026                 8           3
2027                 4           0

Long weekends per year:
start
2023    1
2024    4
2025    7
2026    5
2027    2

Long weekends from November 2026:
     start start_day        end  days                   holidays
2026-12-25       Fri 2026-12-27     3  Boxing Day, Christmas Day
2027-01-01       Fri 2027-01-03     3             New Year's Day
2027-03-26       Fri 2027-03-29     4 Easter Monday, Good Friday

All long weekends 2026:
     start start_day        end  days                   holidays
2026-03-20       Fri 2026-03-22     3    Eid-al-Fitr (estimated)
2026-04-03       Fri 2026-04-06     4 Easter Monday, Good Friday
2026-05-01       Fri 2026-05-03     3                  Labor Day
2026-05-30       Sat 2026-06-01     3               Madaraka Day
2026-12-25       F

## **Paydays, the international seasonal index, and the finished calendar**

We add three more columns:
- `fri_sat_night`: Friday and Saturday nights, the core of weekend short stays.
- `payday_window`: day 25 to day 3 of the month. **Assumption**: salaries land around
  month-end and spending rises. We have no data to measure it.
- `intl_seasonal_index`: the Chapter 1 index (1.0 = average month) mapped to each day.

Then a comparison that matters for design: **international arrivals and domestic
school-holiday pressure do not peak in the same months.** If they differ, the price
advisor cannot use international seasonality alone for homestays that mostly host
Kenyans.

Output: `data/processed/calendar_daily.csv`, one row per day from 1 Jan 2024 to
31 Mar 2027, which the synthetic history generator (Chapter 3) will read.

In [6]:
out_dir = Path("../data/processed")
month_names = [pd.Timestamp(2000, m, 1).strftime("%b") for m in range(1, 13)]

seasonal = pd.read_csv(out_dir / "seasonal_index.csv")
idx_map = dict(zip(seasonal["month_number"], seasonal["seasonal_index"]))

cal = calendar_full.copy()
cal["year"] = cal["date"].dt.year
cal["month"] = cal["date"].dt.month
cal["day_of_month"] = cal["date"].dt.day
cal["intl_seasonal_index"] = cal["month"].map(idx_map)
cal["payday_window"] = ((cal["day_of_month"] >= 25) | (cal["day_of_month"] <= 3)).astype(int)
cal["fri_sat_night"] = cal["dow"].isin([4, 5]).astype(int)

# Checks
print("shape:", cal.shape)
print("date range:", cal["date"].min().date(), "to", cal["date"].max().date())
print("duplicate dates:", int(cal["date"].duplicated().sum()))
missing = cal.isna().sum()
print("columns with missing values:", missing[missing > 0].to_dict() or "none")
print("columns:", list(cal.columns))

cal.to_csv(out_dir / "calendar_daily.csv", index=False)
print("saved:", out_dir / "calendar_daily.csv")

# Two seasonalities side by side, using the three full years 2024-2026
full_years = cal[cal["year"].isin([2024, 2025, 2026])]
monthly = full_years.groupby("month").agg(
    school_out_share=("school_out", "mean"),
    long_weekend_days=("is_long_weekend", "sum"),
).round(2)
monthly["intl_index"] = pd.Series(idx_map).round(2)
monthly.index = month_names
print("\nSeasonality: international arrivals vs school-holiday pressure (2024-2026)")
print(monthly[["intl_index", "school_out_share", "long_weekend_days"]].T.to_string())

# The calendar the advisor will see right after the demo date
window = cal[(cal["date"] >= "2026-11-26") & (cal["date"] <= "2027-01-05")]
print("\nDemo window, 26 Nov 2026 to 5 Jan 2027:")
print(window[["date", "dow", "school_out", "holiday_name", "long_weekend_len",
              "payday_window", "intl_seasonal_index"]]
      .assign(date=window["date"].dt.strftime("%a %d %b"))
      .drop(columns="dow")
      .to_string(index=False))

shape: (1186, 18)
date range: 2024-01-01 to 2027-03-31
duplicate dates: 0
columns with missing values: {'school_break': 786}
columns: ['date', 'school_break', 'school_out', 'holiday_name', 'is_public_holiday', 'holiday_estimated', 'dow', 'is_weekend', 'non_working', 'is_long_weekend', 'long_weekend_len', 'easter_period', 'year', 'month', 'day_of_month', 'intl_seasonal_index', 'payday_window', 'fri_sat_night']
saved: ../data/processed/calendar_daily.csv

Seasonality: international arrivals vs school-holiday pressure (2024-2026)
                    Jan   Feb   Mar   Apr   May   Jun   Jul   Aug   Sep   Oct   Nov    Dec
intl_index         1.02  0.96  0.85  0.79  0.83  1.01  1.17  1.28  1.02  1.02  0.94   1.12
school_out_share   0.17  0.09  0.06  0.78  0.00  0.16  0.00  0.74  0.00  0.23  1.00   1.00
long_weekend_days  1.00  0.00  9.00  9.00  9.00  3.00  0.00  0.00  0.00  9.00  3.00  10.00

Demo window, 26 Nov 2026 to 5 Jan 2027:
      date  school_out   holiday_name  long_weekend_len  payda